# 01 Candidate Generation — FAISS embedding blocking

Этот notebook превращает очищенный category-run срез в пары-кандидаты для ручной разметки и последующего benchmark. Здесь ещё нет финального решения "дубль / не дубль": мы только строим широкий, но управляемый shortlist похожих SKU.

**Результат:** CSV `candidates_<suffix>.csv` с FAISS-соседями, supplemental training coverage pairs и диагностикой источников кандидатов.

## Оглавление

- [0. Локальные настройки](#0-локальные-настройки)
- [1. Подготовка окружения и конфигурация запуска](#1-подготовка-окружения-и-конфигурация-запуска)
- [2. Загрузка project/category среза из DuckDB](#2-загрузка-projectcategory-среза-из-duckdb)
- [3. Product records](#3-product-records)
- [4. Embeddings](#4-embeddings)
- [5. FAISS top-k blocking](#5-faiss-top-k-blocking)
- [6. Проверка FAISS-соседей](#6-проверка-faiss-соседей)
- [7. Сохранение candidate CSV](#7-сохранение-candidate-csv)
- [8. Распределение embedding-score](#8-распределение-embedding-score)
- [9. Sanity checks](#9-sanity-checks)
- [10. Итоговые выводы](#10-итоговые-выводы)


## Ход действий

1. Загружаем выбранный project/category срез из `mpstats_products`.
2. Приводим строки к research-контракту `marketplace + Артикул`, сохраняя `Подкатегория`, brand и готовые weight/pack поля.
3. Собираем embedding-текст из `brand + title` и считаем dense embeddings через `ModelManager`.
4. Ищем top-k соседей через FAISS: основной поиск внутри `Подкатегория`, fallback — global внутри категории.
5. Добавляем supplemental пары, чтобы будущая разметка не зависела только от текущего `top_k`.
6. Сохраняем candidate CSV и проверяем распределения/примеры перед `02_labeling_dataset.ipynb`.


## 0. Локальные настройки

Это главный пульт `01`: категория, DuckDB, embedding-модель, FAISS top-k, supplemental пары и примеры соседей задаются здесь. Секреты для Polza.ai/Hugging Face остаются в `.env`, но обычные параметры запуска — в этой ячейке.


In [ ]:
# === MY notebook settings ===
# Category-run: "sauces", "coconut_oil" или "soap".
MY_CATEGORY_RUN = 'sauces'

# None = искать mpstats.duckdb в корне проекта. Можно поставить абсолютный путь.
MY_DUCKDB_PATH = None

# None = стандартный candidates_<suffix>.csv для выбранного category-run.
MY_CANDIDATES_PATH = None

# Embedding model: alias из registry или прямой model id. Для Polza укажите backend "polza_embedding".
MY_EMBEDDING_MODEL = 'embedding_e5_small'
MY_EMBEDDING_BACKEND = None
MY_EMBEDDING_BATCH_SIZE = 64  # None = batch size из registry

# FAISS retrieval.
MY_FAISS_TOP_K = 30
MY_FAISS_SUBCATEGORY_BLOCKING = True
MY_FAISS_GLOBAL_SAFETY_TOP_K = 5
MY_FAISS_UNKNOWN_SUBCATEGORY_TOP_K = 30
MY_FAISS_MAX_CANDIDATES = 1000000
MY_FAISS_RECORD_LIMIT = None  # None = все строки
MY_FAISS_MIN_SIMILARITY = None  # None = без нижнего порога

# Supplemental training coverage pairs: расширяют candidate CSV за пределы FAISS top-k.
MY_SUPPLEMENTAL_PAIRS_ENABLED = True
MY_SUPPLEMENTAL_LEXICAL_PAIRS = 8000
MY_SUPPLEMENTAL_SAME_BRAND_PACK_PAIRS = 6000
MY_SUPPLEMENTAL_CROSS_MARKETPLACE_RANDOM_PAIRS = 3000
MY_SUPPLEMENTAL_RANDOM_PAIRS = 3000
MY_SUPPLEMENTAL_RANDOM_STATE = 42

# Диагностический блок ближайших соседей в конце notebook.
MY_NEIGHBOR_GROUP_COUNT = 6
MY_NEIGHBOR_GROUP_SIZE = 30
MY_NEIGHBOR_GROUP_SAMPLE_MODE = 'top_dense'  # "top_dense" или "random"
MY_NEIGHBOR_GROUP_RANDOM_STATE = 42


## 1. Подготовка окружения и конфигурация запуска

Подключаем проектные модули, проверяем research-зависимости и собираем объект `FaissCandidateGenerationConfig`. Эта секция не читает данные и не создаёт артефакты.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import time

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.services.sales_filter_service import DEFAULT_SALES_FILTER_GROUP_COLUMNS, DEFAULT_SALES_MIN_QUANTILE, DEFAULT_SALES_MIN_UNITS, filter_sales_by_quantile
from research.dedup import (
    FAISS_CANDIDATE_OUTPUT_COLUMNS,
    CandidateGenerationConfig,
    FaissCandidateGenerationConfig,
    ModelManager,
    POLZA_EMBEDDING_BACKEND,
    generate_faiss_candidate_pairs,
    model_text_prefix,
    prepare_product_records,
    resolve_category_run,
    resolve_run_paths,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)


In [ ]:
CATEGORY_RUN = resolve_category_run(MY_CATEGORY_RUN)
RUN_PATHS = resolve_run_paths(PROJECT_ROOT, CATEGORY_RUN)
TARGET_CATEGORY = CATEGORY_RUN.display_name
CATEGORY_ALIASES = list(CATEGORY_RUN.category_aliases)
PROJECT_NAME = CATEGORY_RUN.project_name
PRODUCTS_TABLE = "mpstats_products"
SALES_MIN_QUANTILE = DEFAULT_SALES_MIN_QUANTILE
SALES_MIN_UNITS = DEFAULT_SALES_MIN_UNITS
SALES_FILTER_DESCRIPTION = (
    f"min sales {SALES_MIN_UNITS:g}"
    if SALES_MIN_QUANTILE is None
    else f"bottom quantile {SALES_MIN_QUANTILE:.0%}; min sales {SALES_MIN_UNITS:g}"
)
DATA_DIR = RUN_PATHS.data_dir
CANDIDATES_PATH = Path(MY_CANDIDATES_PATH).expanduser() if MY_CANDIDATES_PATH else RUN_PATHS.candidates_path

MODEL_MANAGER = ModelManager()
EMBEDDING_MODEL = str(MY_EMBEDDING_MODEL).strip()
EMBEDDING_MODEL_BACKEND = str(MY_EMBEDDING_BACKEND).strip() if MY_EMBEDDING_BACKEND else None
EMBEDDING_MODEL_SPEC = MODEL_MANAGER.resolve_embedding_model(EMBEDDING_MODEL, backend=EMBEDDING_MODEL_BACKEND)
EMBEDDING_BATCH_SIZE = int(MY_EMBEDDING_BATCH_SIZE or EMBEDDING_MODEL_SPEC.batch_size or 64)
FAISS_TOP_K = int(MY_FAISS_TOP_K)
SUBCATEGORY_BLOCKING = bool(MY_FAISS_SUBCATEGORY_BLOCKING)
GLOBAL_SAFETY_TOP_K = int(MY_FAISS_GLOBAL_SAFETY_TOP_K)
UNKNOWN_SUBCATEGORY_TOP_K = int(MY_FAISS_UNKNOWN_SUBCATEGORY_TOP_K)
MAX_CANDIDATES = int(MY_FAISS_MAX_CANDIDATES) if MY_FAISS_MAX_CANDIDATES is not None else None
RECORD_LIMIT = int(MY_FAISS_RECORD_LIMIT) if MY_FAISS_RECORD_LIMIT is not None else None
MIN_SIMILARITY = float(MY_FAISS_MIN_SIMILARITY) if MY_FAISS_MIN_SIMILARITY is not None else None

supplemental_enabled = bool(MY_SUPPLEMENTAL_PAIRS_ENABLED)
feature_config = CandidateGenerationConfig(max_candidates=None)
faiss_config = FaissCandidateGenerationConfig(
    top_k=FAISS_TOP_K,
    max_candidates=MAX_CANDIDATES,
    min_similarity=MIN_SIMILARITY,
    subcategory_blocking=SUBCATEGORY_BLOCKING,
    global_safety_top_k=GLOBAL_SAFETY_TOP_K,
    unknown_subcategory_top_k=UNKNOWN_SUBCATEGORY_TOP_K,
    candidate_features=feature_config,
    supplemental_lexical_pairs=int(MY_SUPPLEMENTAL_LEXICAL_PAIRS) if supplemental_enabled else 0,
    supplemental_same_brand_pack_pairs=int(MY_SUPPLEMENTAL_SAME_BRAND_PACK_PAIRS) if supplemental_enabled else 0,
    supplemental_cross_marketplace_random_pairs=int(MY_SUPPLEMENTAL_CROSS_MARKETPLACE_RANDOM_PAIRS) if supplemental_enabled else 0,
    supplemental_random_pairs=int(MY_SUPPLEMENTAL_RANDOM_PAIRS) if supplemental_enabled else 0,
    supplemental_random_state=int(MY_SUPPLEMENTAL_RANDOM_STATE),
)


def resolve_duckdb_path(project_root: Path) -> Path:
    '''Возвращает существующий DuckDB-куб для чтения candidate source rows.'''
    explicit_path = Path(MY_DUCKDB_PATH).expanduser() if MY_DUCKDB_PATH else None
    candidates = [explicit_path, project_root / "mpstats.duckdb"]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "DuckDB-куб не найден. Укажите MY_DUCKDB_PATH в первой code-ячейке "
        "или положите mpstats.duckdb в корень проекта."
    )


missing = [
    package
    for package, module_name in {"faiss-cpu": "faiss", "sentence-transformers": "sentence_transformers"}.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    raise ImportError(
        "Для FAISS embedding blocking нужны research-зависимости: "
        + ", ".join(missing)
        + ". Установите: python3 -m pip install -r requirements-research.txt"
    )

# Import FAISS before sentence-transformers/torch. On local macOS/Python 3.13,
# importing FAISS after torch can crash the process in native code.
import faiss

DB_PATH = resolve_duckdb_path(PROJECT_ROOT)
print(f"Category run: {CATEGORY_RUN.slug} — {TARGET_CATEGORY}; project: {PROJECT_NAME}")
print(f"DuckDB cube: {DB_PATH}")
print(f"Embedding model alias/input: {EMBEDDING_MODEL}")
print(f"Embedding model id: {EMBEDDING_MODEL_SPEC.model_name}")
print(f"Embedding backend: {EMBEDDING_MODEL_SPEC.backend}")
if EMBEDDING_MODEL_SPEC.backend == POLZA_EMBEDDING_BACKEND:
    print(f"Polza base URL: {MODEL_MANAGER.polza_base_url}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")
print(f"FAISS top_k: {FAISS_TOP_K}; max candidates: {MAX_CANDIDATES}; min similarity: {MIN_SIMILARITY}")
print(
    "Subcategory blocking: "
    f"{SUBCATEGORY_BLOCKING}; global safety top_k: {GLOBAL_SAFETY_TOP_K}; "
    f"unknown-subcategory top_k: {UNKNOWN_SUBCATEGORY_TOP_K}"
)
print(
    "Supplemental pairs: "
    f"enabled={supplemental_enabled}; lexical={faiss_config.supplemental_lexical_pairs}; "
    f"same_brand_pack={faiss_config.supplemental_same_brand_pack_pairs}; "
    f"cross_marketplace_random={faiss_config.supplemental_cross_marketplace_random_pairs}; "
    f"random={faiss_config.supplemental_random_pairs}"
)
print(f"Record limit: {RECORD_LIMIT}")
print(f"Candidates output: {CANDIDATES_PATH}")


## 2. Загрузка project/category среза из DuckDB

Читаем только выбранный project/category из `mpstats_products`, затем применяем общий sales-фильтр `Продажи >= 15`, чтобы candidate generation не раздувался хвостом слабых продаж.


In [ ]:

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    columns = set(con.execute(f'DESCRIBE {PRODUCTS_TABLE}').fetchdf()["column_name"].astype(str))
    project_filter_sql = ""
    project_filter_params: list[str] = []
    if PROJECT_NAME:
        if "__project_name" not in columns:
            raise RuntimeError("Для category-run нужен project-фильтр, но в кубе нет __project_name.")
        project_filter_sql = 'WHERE "__project_name" = ?'
        project_filter_params.append(PROJECT_NAME)

    available_categories = con.execute(
        f'SELECT DISTINCT "Категория" FROM {PRODUCTS_TABLE} {project_filter_sql} ORDER BY 1',
        project_filter_params,
    ).fetchdf()["Категория"].dropna().tolist()
    real_category = next((category for category in CATEGORY_ALIASES if category in available_categories), None)
    if real_category is None:
        raise ValueError(
            f"Категория {TARGET_CATEGORY!r} не найдена в проекте {PROJECT_NAME!r}. "
            f"Доступно: {available_categories[:20]}"
        )

    where_clauses = ['"Категория" = ?']
    params: list[object] = [real_category]
    if PROJECT_NAME:
        where_clauses.append('"__project_name" = ?')
        params.append(PROJECT_NAME)
    where_sql = " AND ".join(where_clauses)
    products_df = con.execute(
        f'SELECT * FROM {PRODUCTS_TABLE} WHERE {where_sql}',
        params,
    ).fetchdf()

rows_before_sales_filter = len(products_df)
products_df = filter_sales_by_quantile(
    products_df,
    sales_column="Продажи, шт",
    quantile=SALES_MIN_QUANTILE,
    min_sales=SALES_MIN_UNITS,
    group_columns=DEFAULT_SALES_FILTER_GROUP_COLUMNS,
).reset_index(drop=True)

print(f"Resolved project/category: {PROJECT_NAME!r} / {real_category!r}")
print(f"Loaded rows after sales filter: {len(products_df):,} / {rows_before_sales_filter:,} ({SALES_FILTER_DESCRIPTION})")
display(products_df.head(3))


## 3. Product records

Один record — это `marketplace + Артикул`. Повторы по месяцам внутри одного marketplace агрегируются, чтобы FAISS искал соседей между товарами, а не между месячными строками одного товара.

In [ ]:
product_records = prepare_product_records(products_df, feature_config)
if RECORD_LIMIT is not None:
    product_records = product_records.head(RECORD_LIMIT).copy()
    print(f"Using first {len(product_records):,} product records because MY_FAISS_RECORD_LIMIT is set")
articles_by_marketplaces = product_records.groupby("sku")["marketplace"].nunique(dropna=True)
summary = pd.DataFrame(
    [
        {
            "raw_rows": len(products_df),
            "product_records": len(product_records),
            "unique_articles": product_records["sku"].nunique(dropna=True),
            "marketplaces": product_records["marketplace"].nunique(dropna=True),
            "articles_seen_in_multiple_marketplaces": int((articles_by_marketplaces > 1).sum()),
            "unique_titles": product_records["title_norm"].nunique(dropna=True),
            "brand_fill_share": product_records["brand_norm"].ne("").mean(),
            "subcategory_fill_share": product_records["subcategory_norm"].ne("").mean(),
            "unique_subcategories": product_records["subcategory_norm"].replace("", pd.NA).nunique(dropna=True),
            "multipack_gt_1_share": (pd.to_numeric(product_records["multipack_count"], errors="coerce") > 1).mean(),
        }
    ]
)
display(summary)
display(product_records.head(5))

## 4. Embeddings

Embedding-текст собирается из бренда и title. Для E5-моделей добавляется `query:` prefix: у нас SKU сравнивается с SKU, то есть это symmetric similarity, а не web query -> passage retrieval.

In [ ]:
def build_embedding_text(row: pd.Series) -> str:
    '''Собирает короткий текст для embedding-модели из brand и title.'''
    parts = [str(row.get("brand") or "").strip(), str(row.get("title") or "").strip()]
    text = " ".join(part for part in parts if part)
    prefix = model_text_prefix(EMBEDDING_MODEL)
    return f"{prefix or ''}{text}".strip()

embedding_texts = product_records.apply(build_embedding_text, axis=1).tolist()
display(pd.DataFrame({"embedding_text": embedding_texts[:5]}))

model = MODEL_MANAGER.load_embedding_model(EMBEDDING_MODEL, backend=EMBEDDING_MODEL_BACKEND)
started_at = time.perf_counter()
embeddings = model.encode(
    embedding_texts,
    batch_size=EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
elapsed_sec = time.perf_counter() - started_at
print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding time: {elapsed_sec:.2f} sec")

## 5. FAISS top-k blocking

Здесь и происходит первичный поиск кандидатов: FAISS ищет top-k ближайших embedding-соседей для каждого product record. Если `Подкатегория` заполнена хотя бы у части строк, основной поиск идёт внутри одной подкатегории; пустые подкатегории и маленький global safety-net ищут по всему срезу. Если подкатегорий нет вообще, notebook автоматически работает в старом full-global режиме. `baseline_similarity_score` оставлен только как совместимое имя для следующих ноутбуков; его значение равно `embedding_similarity_score`.

In [ ]:
started_at = time.perf_counter()
candidates = generate_faiss_candidate_pairs(product_records, embeddings, faiss_config, faiss_module=faiss)
elapsed_sec = time.perf_counter() - started_at

print(f"Candidate pairs: {len(candidates):,}")
print(f"FAISS generation time: {elapsed_sec:.2f} sec")
display(candidates.head(10))

## 6. Проверка FAISS-соседей

Эта диагностика показывает несколько anchor-SKU и их ближайших FAISS-соседей из текущего candidate shortlist. Удобно глазами оценить, хватает ли текущего `FAISS_TOP_K` или нужно увеличить `MY_FAISS_TOP_K`.

In [ ]:
NEIGHBOR_GROUP_COUNT = int(MY_NEIGHBOR_GROUP_COUNT)
NEIGHBOR_GROUP_SIZE = int(MY_NEIGHBOR_GROUP_SIZE or FAISS_TOP_K)
NEIGHBOR_GROUP_SAMPLE_MODE = str(MY_NEIGHBOR_GROUP_SAMPLE_MODE or "top_dense")  # top_dense | random
NEIGHBOR_GROUP_RANDOM_STATE = int(MY_NEIGHBOR_GROUP_RANDOM_STATE)


def _compact_values(value: object, limit: int = 8) -> str:
    '''Сжимает список исходных значений до читаемой строки для диагностики соседей.'''
    if isinstance(value, (list, tuple, set)):
        values = [str(item) for item in value if str(item).strip()]
    elif pd.isna(value):
        values = []
    else:
        values = [str(value)] if str(value).strip() else []
    if len(values) > limit:
        return " | ".join(values[:limit]) + f" | ... (+{len(values) - limit})"
    return " | ".join(values)


def _record_summary(
    raw_record_id: str,
    *,
    row_type: str,
    neighbor_rank: int | None = None,
    score: float | None = None,
    blocking_scope: str | None = None,
    subcategory_relation: str | None = None,
) -> dict[str, object]:
    '''Возвращает одну строку anchor/neighbor summary для визуальной проверки FAISS.'''
    record = records_by_id.loc[raw_record_id]
    return {
        "row_type": row_type,
        "neighbor_rank": neighbor_rank,
        "score": score,
        "raw_record_id": raw_record_id,
        "marketplace": record.get("marketplace"),
        "sku": record.get("sku"),
        "brand": record.get("brand"),
        "subcategory": record.get("subcategory"),
        "title": record.get("title"),
        "blocking_scope": blocking_scope,
        "subcategory_relation": subcategory_relation,
        "unit_amount": record.get("unit_amount"),
        "total_amount": record.get("total_amount"),
        "multipack_count": record.get("multipack_count"),
        "collapsed_record_count": record.get("collapsed_record_count", 1),
        "source_skus": _compact_values(record.get("source_skus", record.get("sku"))),
        "marketplaces": _compact_values(record.get("marketplaces", record.get("marketplace"))),
    }


records_by_id = product_records.set_index("raw_record_id", drop=False)
left_edges = candidates.rename(
    columns={
        "raw_record_id_a": "anchor_id",
        "raw_record_id_b": "neighbor_id",
        "sku_a": "anchor_sku",
        "sku_b": "neighbor_sku",
    }
)
right_edges = candidates.rename(
    columns={
        "raw_record_id_b": "anchor_id",
        "raw_record_id_a": "neighbor_id",
        "sku_b": "anchor_sku",
        "sku_a": "neighbor_sku",
    }
)
neighbor_edges = pd.concat(
    [
        left_edges[["anchor_id", "neighbor_id", "embedding_similarity_score", "candidate_rank", "blocking_scope", "subcategory_relation"]],
        right_edges[["anchor_id", "neighbor_id", "embedding_similarity_score", "candidate_rank", "blocking_scope", "subcategory_relation"]],
    ],
    ignore_index=True,
).drop_duplicates(["anchor_id", "neighbor_id"])
neighbor_edges = neighbor_edges[neighbor_edges["anchor_id"].isin(records_by_id.index)]
neighbor_edges = neighbor_edges[neighbor_edges["neighbor_id"].isin(records_by_id.index)]
neighbor_edges = neighbor_edges.sort_values(
    ["embedding_similarity_score", "candidate_rank", "anchor_id", "neighbor_id"],
    ascending=[False, True, True, True],
).reset_index(drop=True)

anchor_counts = neighbor_edges.groupby("anchor_id", as_index=False).size().rename(columns={"size": "neighbor_count"})
if NEIGHBOR_GROUP_SAMPLE_MODE == "random":
    selected_anchors = anchor_counts.sample(
        n=min(NEIGHBOR_GROUP_COUNT, len(anchor_counts)),
        random_state=NEIGHBOR_GROUP_RANDOM_STATE,
    )
else:
    selected_anchors = anchor_counts.sort_values(
        ["neighbor_count", "anchor_id"],
        ascending=[False, True],
    ).head(NEIGHBOR_GROUP_COUNT)

print(
    f"Showing {len(selected_anchors)} FAISS neighbor groups; "
    f"mode={NEIGHBOR_GROUP_SAMPLE_MODE}; group_size={NEIGHBOR_GROUP_SIZE}; "
    f"candidate rows={len(candidates):,}."
)
print(
    "neighbor_count считается после удаления зеркальных пар и глобального MAX_CANDIDATES; "
    "если группам часто не хватает соседей, проверьте MY_FAISS_TOP_K и MY_FAISS_MAX_CANDIDATES."
)

for group_number, anchor in enumerate(selected_anchors.itertuples(index=False), start=1):
    anchor_id = str(anchor.anchor_id)
    neighbors = neighbor_edges[neighbor_edges["anchor_id"].eq(anchor_id)].head(NEIGHBOR_GROUP_SIZE)
    rows = [_record_summary(anchor_id, row_type="anchor", neighbor_rank=None, score=None)]
    for rank, edge in enumerate(neighbors.itertuples(index=False), start=1):
        rows.append(
            _record_summary(
                str(edge.neighbor_id),
                row_type="neighbor",
                neighbor_rank=rank,
                score=round(float(edge.embedding_similarity_score), 6),
                blocking_scope=edge.blocking_scope,
                subcategory_relation=edge.subcategory_relation,
            )
        )
    print()
    print(f"Group {group_number}: anchor={anchor_id}; available_neighbors={int(anchor.neighbor_count)}")
    display(pd.DataFrame(rows))


## 7. Сохранение candidate CSV

Сохраняем только стабильные колонки `FAISS_CANDIDATE_OUTPUT_COLUMNS`, чтобы следующий notebook работал с предсказуемым контрактом.


In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
candidates_to_save = candidates[FAISS_CANDIDATE_OUTPUT_COLUMNS].copy()
candidates_to_save.to_csv(CANDIDATES_PATH, index=False)

print(f"Saved candidates: {CANDIDATES_PATH}")
print(f"Rows saved: {len(candidates_to_save):,}")
display(candidates_to_save.head(5))

pair_scope_stats = (
    candidates_to_save.assign(
        pair_scope=candidates_to_save["is_cross_marketplace_pair"].map(
            {True: "cross_marketplace", False: "same_marketplace"}
        )
    )["pair_scope"]
    .value_counts()
    .rename_axis("pair_scope")
    .reset_index(name="pairs")
)
display(pair_scope_stats)

blocking_scope_stats = (
    candidates_to_save["blocking_scope"]
    .value_counts(dropna=False)
    .rename_axis("blocking_scope")
    .reset_index(name="pairs")
)
display(blocking_scope_stats)

subcategory_relation_stats = (
    candidates_to_save["subcategory_relation"]
    .value_counts(dropna=False)
    .rename_axis("subcategory_relation")
    .reset_index(name="pairs")
)
display(subcategory_relation_stats)

## 8. Распределение embedding-score

In [ ]:
score_summary = candidates_to_save["embedding_similarity_score"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).to_frame("embedding_similarity_score")
display(score_summary)

ax = candidates_to_save["embedding_similarity_score"].hist(bins=40, figsize=(10, 4), color="#4C78A8")
ax.set_title("Distribution of FAISS embedding cosine score")
ax.set_xlabel("embedding_similarity_score")
ax.set_ylabel("candidate pairs")
plt.tight_layout()
plt.show()

## 9. Sanity checks

Hard-negative и pack-variant флаги ниже не создают пары. Они только помечают уже найденные FAISS пары, чтобы `02_labeling_dataset.ipynb` мог выбрать полезную ручную разметку.

In [ ]:
flag_stats = pd.DataFrame(
    [
        {"flag": "is_cross_marketplace_pair", "pairs": int(candidates_to_save["is_cross_marketplace_pair"].fillna(False).sum())},
        {"flag": "is_hard_negative_candidate", "pairs": int(candidates_to_save["is_hard_negative_candidate"].fillna(False).sum())},
        {"flag": "is_pack_variant_candidate", "pairs": int(candidates_to_save["is_pack_variant_candidate"].fillna(False).sum())},
    ]
)
display(flag_stats)

def show_examples(frame: pd.DataFrame, label: str, n: int = 5) -> None:
    '''Показывает первые пары выбранного типа в стабильном наборе колонок.'''
    if frame.empty:
        print(f"{label}: нет примеров")
        return
    print(label)
    display(
        frame[[
            "raw_record_id_a",
            "raw_record_id_b",
            "marketplace_a",
            "marketplace_b",
            "sku_a",
            "sku_b",
            "title_a",
            "title_b",
            "brand_a",
            "brand_b",
            "subcategory_a",
            "subcategory_b",
            "subcategory_relation",
            "embedding_similarity_score",
            "candidate_rank",
            "blocking_scope",
            "is_cross_marketplace_pair",
            "is_hard_negative_candidate",
            "is_pack_variant_candidate",
        ]].head(n)
    )

show_examples(candidates_to_save.head(20), "Top FAISS candidates")
show_examples(candidates_to_save[candidates_to_save["is_cross_marketplace_pair"]], "Cross-marketplace candidates")
show_examples(candidates_to_save[candidates_to_save["is_hard_negative_candidate"]], "Hard-negative candidates")
show_examples(candidates_to_save[candidates_to_save["is_pack_variant_candidate"]], "Pack-variant candidates")

## 10. Итоговые выводы

- `candidates_<suffix>.csv` теперь строится через dense embeddings + FAISS top-k.
- Если `Подкатегория` заполнена, основной top-k считается внутри неё; пустые подкатегории и `global_safety` остаются full-global, а полностью пустой срез автоматически откатывается в старый full-global режим.
- `baseline_similarity_score` в этом файле — это совместимое имя для embedding cosine score.
- Следующий шаг: запустить `02_labeling_dataset.ipynb`, разметить `label`, затем сравнивать методы pairwise matching в `03_matching_comparison.ipynb`.